In [2]:
# 1. Importing Libraries
import sys
from pathlib import Path

# 2. Setting project paths
project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

# Define standard data subdirectories for easy access later
RAW_DATA_DIR = project_root / "data" / "raw"
PROCESSED_DATA_DIR = project_root / "data" / "processed"
ASSETS_DIR = project_root / "assets" / "3D_Objects"


# Read Data (AKR Burst + Residence Time)

In [ ]:
# 1. Importing libraries
import pandas as pd

# 2. Importing parquet file
wind_data = pd.read_parquet(
    f"{project_root}/data/processed/01_processed_wind_data_fogg_akr_burst_list_1995_2004.parquet",
)
wind_data.head(10)


In [ ]:
# cols_to_round = ["LT_gse", "radius", "lat_gse", "lon_gse", "x_gse", "y_gse", "z_gse"]

# wind_data_copy = wind_data.copy()
# wind_data_copy[cols_to_round] = wind_data_copy[cols_to_round].round(2)

# wind_data_copy = wind_data_copy.drop_duplicates(
#     subset=cols_to_round,
#     keep="first",
# )

# merged_df = wind_data_copy.merge(residence_data, on=cols_to_round, how="outer")

# wind_data_copy.head(10)


In [ ]:
# output_path = Path(f"{RAW_DATA_DIR}/residence_data.parquet")
residence_data = pd.read_parquet(f"{RAW_DATA_DIR}/residence_data.parquet")
residence_data.head(10)


In [ ]:
# residence_data.loc[residence_data["x_gse"] == 234, :]

residence_data.loc[residence_data["x_gse"] >= 234, :]


In [ ]:
# Check if x y z in correct order


# Connect them with 0 and 1

In [ ]:
cols_to_round = ["LT_gse", "radius", "lat_gse", "lon_gse", "x_gse", "y_gse", "z_gse"]

merged_df = (
    wind_data.assign(**{col: lambda df, c=col: df[c].round(2) for col in cols_to_round})
    .drop_duplicates(subset=cols_to_round, keep="first")
    .merge(
        # Round residence_data on-the-fly to ensure keys match
        residence_data.assign(
            **{col: lambda df, c=col: df[c].round(2) for col in cols_to_round}
        ),
        on=cols_to_round,
        how="outer",
    )
)


In [ ]:
processed_data_base_name = "03_merged_data_for_modelling"
merged_df.to_parquet(
    f"{PROCESSED_DATA_DIR}/{processed_data_base_name}.parquet",
    engine="pyarrow",
    index=False,
)


# Modeling with GPFLow

## Importing Data

In [3]:
import pandas as pd

processed_data_base_name = "03_merged_data_for_modelling"
merged_df = pd.read_parquet(f"{PROCESSED_DATA_DIR}/{processed_data_base_name}.parquet")
merged_df.head(10)


,original_burst_id,stime,etime,burst_timestamp,min_f_bound,max_f_bound,LT_gse,radius,lat_gse,lon_gse,x_gse,y_gse,z_gse,time_stamp,x_gsm,y_gsm,z_gsm,lat_gsm,lon_gsm
0,NaN,NaT,NaT,NaT,NaN,NaN,0.0,8.24,-0.76,180.06,-8.24,-0.01,-0.11,2000-07-23 14:00:00,-8.24,-0.05,-0.10,-0.69,180.33
1,1698.0,1998-07-02 14:55:17.365806336,1998-07-02 19:17:58.068891392,1998-07-02 15:34:59.797667968,224.0,940.0,0.0,11.79,1.83,180.07,-11.78,-0.01,0.38,NaT,NaN,NaN,NaN,NaN,NaN
2,1156.0,1997-09-08 19:10:43.067714816,1997-09-08 23:02:51.131067904,1997-09-08 21:12:53.627374336,72.0,1040.0,0.0,14.01,0.98,180.02,-14.01,-0.00,0.24,NaT,NaN,NaN,NaN,NaN,NaN
3,NaN,NaT,NaT,NaT,NaN,NaN,0.0,18.19,43.81,180.08,-13.13,-0.02,12.60,1999-02-05 10:00:00,-13.13,-1.48,12.51,43.43,186.43
4,NaN,NaT,NaT,NaT,NaN,NaN,0.0,20.58,-30.32,180.00,-17.77,-0.00,-10.39,1999-10-30 18:24:00,-17.77,-2.54,-10.08,-29.31,188.13
5,2376.0,1999-02-23 03:47:08.821348864,1999-02-23 07:08:51.272523264,1999-02-23 07:05:47.902050944,272.0,740.0,0.0,21.05,41.24,180.02,-15.83,-0.01,13.88,NaT,NaN,NaN,NaN,NaN,NaN
6,2518.0,1999-03-12 16:13:49.757172224,1999-03-12 17:36:17.884770048,1999-03-12 16:19:56.285142400,196.0,740.0,0.0,23.79,32.03,180.08,-20.18,-0.02,12.62,NaT,NaN,NaN,NaN,NaN,NaN
7,NaN,NaT,NaT,NaT,NaN,NaN,0.0,23.89,5.50,180.02,-23.78,-0.01,2.29,1996-12-31 08:48:00,-23.78,0.23,2.28,5.47,179.44
8,NaN,NaT,NaT,NaT,NaN,NaN,0.0,24.91,-4.51,180.06,-24.83,-0.03,-1.96,2001-09-10 20:24:00,-24.83,-0.51,-1.89,-4.35,181.18
9,3599.0,1999-12-18 16:16:04.842753152,1999-12-18 18:51:56.736950912,1999-12-18 16:58:52.029395712,148.0,428.0,0.0,35.28,-23.28,180.03,-32.41,-0.02,-13.94,NaT,NaN,NaN,NaN,NaN,NaN


In [ ]:
# merged_df.loc[merged_df["x_gse"] >= 234, :]


,original_burst_id,stime,etime,burst_timestamp,min_f_bound,max_f_bound,LT_gse,radius,lat_gse,lon_gse,x_gse,y_gse,z_gse,time_stamp,x_gsm,y_gsm,z_gsm,lat_gsm,lon_gsm
264273,NaN,NaT,NaT,NaT,NaN,NaN,10.50,253.30,1.95,337.57,234.00,-96.58,8.60,2008-07-21 21:24:00,234.00,-96.24,11.83,2.68,337.64
264274,NaN,NaT,NaT,NaT,NaN,NaN,10.50,253.30,1.95,337.58,234.01,-96.57,8.61,2008-07-21 21:12:00,234.01,-96.18,12.16,2.75,337.66
264303,NaN,NaT,NaT,NaT,NaN,NaN,10.50,253.40,2.21,337.54,234.01,-96.73,9.78,2009-07-11 16:36:00,234.01,-93.94,25.08,5.68,338.13
264304,NaN,NaT,NaT,NaT,NaN,NaN,10.50,253.40,2.21,337.54,234.01,-96.72,9.79,2009-07-11 16:24:00,234.01,-93.66,26.07,5.91,338.19
264306,NaN,NaT,NaT,NaT,NaN,NaN,10.50,253.41,2.21,337.55,234.02,-96.71,9.79,2009-07-11 16:12:00,234.02,-93.37,27.06,6.13,338.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690322,NaN,NaT,NaT,NaT,NaN,NaN,13.51,253.87,2.72,22.59,234.13,97.40,12.06,2005-06-06 06:48:00,234.13,97.23,13.39,3.02,22.55
690323,NaN,NaT,NaT,NaT,NaN,NaN,13.51,253.89,2.72,22.58,234.16,97.39,12.06,2005-06-06 07:12:00,234.16,97.42,11.79,2.66,22.59
690324,NaN,NaT,NaT,NaT,NaN,NaN,13.51,253.89,2.72,22.59,234.15,97.40,12.06,2005-06-06 07:00:00,234.15,97.33,12.57,2.84,22.57
690325,NaN,NaT,NaT,NaT,NaN,NaN,13.51,253.90,2.72,22.58,234.17,97.38,12.07,2005-06-06 07:24:00,234.17,97.51,11.03,2.49,22.61


## Preaparing for Classification

In [4]:
from pandas import DataFrame

cols = ["original_burst_id", "LT_gse", "radius", "lat_gse", "x_gse", "y_gse", "z_gse"]

class_data: DataFrame = (
    merged_df[cols]
    .copy()
    .assign(AKR_Observed=lambda df: df["original_burst_id"].notna().astype(int))
    .drop(columns=["original_burst_id"])
)

processed_data_base_name = "04_prepared_data_for_classification"
class_data.to_parquet(f"{PROCESSED_DATA_DIR}/{processed_data_base_name}.parquet")

# class_data = merged_df[cols]
# class_data["AKR_Observed"] = [0 if pd.isna(value) else 1 for value in class_data["original_burst_id"] ]


In [ ]:
class_data.loc[class_data["AKR_Observed"] == 1, :].head(10)


,LT_gse,radius,lat_gse,x_gse,y_gse,z_gse,AKR_Observed
1,0.0,11.79,1.83,-11.78,-0.01,0.38,1
2,0.0,14.01,0.98,-14.01,-0.00,0.24,1
5,0.0,21.05,41.24,-15.83,-0.01,13.88,1
6,0.0,23.79,32.03,-20.18,-0.02,12.62,1
9,0.0,35.28,-23.28,-32.41,-0.02,-13.94,1
11,0.0,35.37,19.04,-33.43,-0.00,11.54,1
12,0.0,35.40,19.01,-33.47,-0.02,11.53,1
13,0.0,35.44,18.99,-33.51,-0.03,11.53,1
15,0.0,55.50,-4.97,-55.29,-0.01,-4.81,1
16,0.0,55.52,-4.96,-55.31,-0.02,-4.81,1


In [ ]:
print(
    "Total number of AKR data points", len(class_data[class_data["AKR_Observed"] == 1])
)
print(
    "Total number of AKR data points", len(class_data[class_data["AKR_Observed"] == 0])
)


Total number of AKR data points 217468
Total number of AKR data points 667643


In [ ]:
class_data.dtypes


LT_gse          float64
radius          float64
lat_gse         float64
x_gse           float64
y_gse           float64
z_gse           float64
AKR_Observed      int64
dtype: object